# Sesión 6 · La cifra y su margen
**Análisis Estadístico de la Información — Especialización en Big Data (UCompensar)**

Este cuaderno (*notebook*) acompaña las [diapositivas de la sesión](https://jotamao1985.github.io/ANaLISIS-ESTADiSTICO-DE-LA-INFORMACIoN_Ucomp/diapositivas/sesion-6-la-cifra-y-su-margen.html).
Cada celda lleva entre corchetes el mismo identificador que el código proyectado, así que puedes seguir
la clase celda a celda. La portada del curso está en https://jotamao1985.github.io/ANaLISIS-ESTADiSTICO-DE-LA-INFORMACIoN_Ucomp/
y el enunciado del proyecto final, el tablero, en
[su página](https://jotamao1985.github.io/ANaLISIS-ESTADiSTICO-DE-LA-INFORMACIoN_Ucomp/proyecto-tablero.html).

1. **Archivo → Guardar una copia en Drive** para que tus cambios no se pierdan.
2. Ejecuta las celdas **en orden** con **Mayús + Enter**. Varias usan resultados de las anteriores.
   Si Colab avisa que el cuaderno no es de Google, elige **Ejecutar de todos modos**.
3. Si Colab se reinicia o una celda falla al repetirla, usa **Entorno de ejecución → Ejecutar anteriores**.
4. La primera celda crea los datos del caso *Tingua Domicilios S.A.S.*, una empresa **ficticia**
   (sus datos se inventan con una semilla fija y no corresponden a personas ni a empresas reales),
   y les da a las gráficas los colores del curso.

In [ ]:
#@title ▶ Crea los datos del caso (ejecútala sin modificar) { display-mode: "form" }
"""Datos del caso «Tingua Domicilios S.A.S.» (empresa FICTICIA).

Genera datos/pedidos.csv: los 12 000 pedidos que la plataforma entregó entre el
1 de enero y el 30 de junio de 2026 en cinco ciudades. Todo se inventa con una
semilla fija: los datos no corresponden a personas ni a empresas reales.

Solo usa la biblioteca estándar y, del generador aleatorio, solo random(), que
es lo único que Python garantiza igual entre versiones. Por eso la tabla sale
idéntica en Colab y en cualquier computador.

El caso sirve para todo el curso: variables de todos los tipos (sesión 1),
asimetrías y algunos atípicos (medidas y datos atípicos), una población
completa de la que se pueden sacar muestras (muestreo) y un tiempo de entrega
que depende de la distancia, la lluvia y otras variables (regresión).
"""

import csv
import math
import random
from datetime import date, timedelta
from pathlib import Path

SEMILLA = 2026
N_PEDIDOS = 12_000
INICIO = date(2026, 1, 1)
DIAS = 181  # del 1 de enero al 30 de junio de 2026

COLUMNAS = ["id_pedido", "fecha", "franja", "ciudad", "categoria", "n_productos",
            "valor_pedido", "medio_pago", "estrato", "suscriptor", "distancia_km",
            "lluvia", "temperatura_c", "tiempo_entrega_min", "calificacion"]

# ciudad: (peso, temperatura media °C, desviación, prob. de lluvia, minutos extra, factor de distancia)
CIUDADES = {
    "Bogotá":       (0.40, 14.5, 2.2, 0.30, 5.0, 1.15),
    "Medellín":     (0.22, 22.5, 2.3, 0.28, 2.0, 1.00),
    "Cali":         (0.16, 25.0, 2.3, 0.22, 0.0, 1.00),
    "Barranquilla": (0.12, 29.0, 1.8, 0.12, -1.0, 0.95),
    "Bucaramanga":  (0.10, 23.5, 2.2, 0.20, -2.0, 0.90),
}
LLUVIA_MES = {1: 0.6, 2: 0.7, 3: 1.0, 4: 1.4, 5: 1.3, 6: 0.8}

# franja: (peso, ajuste de temperatura °C, minutos extra)
FRANJAS = {
    "Mañana":   (0.15, -1.5, 0.0),
    "Almuerzo": (0.35, 1.5, 4.0),
    "Tarde":    (0.20, 1.0, 0.0),
    "Noche":    (0.30, -2.0, 3.0),
}

# categoría: (productos extra en promedio, precio medio por producto en COP, minutos de preparación)
CATEGORIAS = {
    "Restaurante": (1.3, 27_000, 3.0),
    "Mercado":     (6.0, 8_500, 4.0),
    "Farmacia":    (0.9, 21_000, 0.0),
    "Mascotas":    (1.2, 36_000, 1.0),
    "Floristería": (0.2, 88_000, 2.0),
}
# Qué se pide en cada franja (mismo orden que CATEGORIAS)
CATEGORIA_POR_FRANJA = {
    "Mañana":   (0.30, 0.35, 0.20, 0.10, 0.05),
    "Almuerzo": (0.78, 0.08, 0.07, 0.04, 0.03),
    "Tarde":    (0.40, 0.25, 0.15, 0.12, 0.08),
    "Noche":    (0.70, 0.10, 0.12, 0.05, 0.03),
}
ESTRATOS = {1: 0.06, 2: 0.22, 3: 0.33, 4: 0.20, 5: 0.11, 6: 0.08}


class _Azar:
    """Lo que hace falta del azar, construido solo sobre random()."""

    def __init__(self, semilla):
        self.r = random.Random(semilla)

    def u(self):
        return self.r.random()

    def elegir(self, opciones, pesos):
        x, acum = self.u() * sum(pesos), 0.0
        for opcion, peso in zip(opciones, pesos):
            acum += peso
            if x < acum:
                return opcion
        return opciones[-1]

    def normal(self, media=0.0, desv=1.0):
        # Box-Muller
        u1, u2 = 1.0 - self.u(), self.u()
        return media + desv * math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)

    def exponencial(self, media):
        return -media * math.log(1.0 - self.u())

    def poisson(self, lam):
        # Knuth: suficiente para medias pequeñas
        limite, k, p = math.exp(-lam), 0, 1.0
        while True:
            p *= self.u()
            if p <= limite:
                return k
            k += 1


def _redondear(x, decimales=0):
    factor = 10 ** decimales
    return math.floor(x * factor + 0.5) / factor


def _pedido(az, dias, pesos_dias):
    fecha = az.elegir(dias, pesos_dias)
    ciudad = az.elegir(list(CIUDADES), [v[0] for v in CIUDADES.values()])
    _, t_media, t_desv, p_lluvia, min_ciudad, f_dist = CIUDADES[ciudad]
    franja = az.elegir(list(FRANJAS), [v[0] for v in FRANJAS.values()])
    _, t_franja, min_franja = FRANJAS[franja]
    categoria = az.elegir(list(CATEGORIAS), CATEGORIA_POR_FRANJA[franja])
    extra, precio, min_prep = CATEGORIAS[categoria]
    estrato = az.elegir(list(ESTRATOS), list(ESTRATOS.values()))

    suscriptor = az.u() < 0.10 + 0.06 * (estrato - 1)
    p_efectivo = 0.45 - 0.07 * (estrato - 1)
    medio_pago = az.elegir(["Efectivo", "Billetera digital", "Tarjeta"],
                           [p_efectivo, 0.25, 1 - p_efectivo - 0.25])

    n = 1 + az.poisson(extra)
    if categoria in ("Mercado", "Restaurante") and az.u() < 0.003:
        n = 20 + int(az.u() * 26)  # pedido corporativo: pocos, pero existen
    valor = n * precio * math.exp(az.normal(0, 0.22)) * (1 + 0.07 * (estrato - 3))
    valor = max(3_000, _redondear(valor / 100) * 100)

    distancia = math.exp(az.normal(math.log(3.0), 0.5)) * f_dist
    distancia = min(14.0, max(0.4, _redondear(distancia, 1)))

    lluvia = az.u() < p_lluvia * LLUVIA_MES[fecha.month]
    temperatura = _redondear(az.normal(t_media + t_franja - 1.8 * lluvia, t_desv), 1)

    tiempo = (11 + 3.1 * distancia + 0.45 * min(n, 15) + min_prep + min_franja + min_ciudad
              + 7.5 * lluvia - 2.5 * suscriptor + az.normal(0, 4) + az.exponencial(3))
    if az.u() < 0.005:
        tiempo += 55 + 95 * az.u()  # pedidos que se demoraron muchísimo
    tiempo = max(6.0, _redondear(tiempo, 1))

    latente = 6.4 - 0.065 * tiempo + 0.2 * suscriptor + az.normal(0, 0.6)
    estrellas = int(min(5, max(1, math.floor(latente + 0.5))))
    califico = az.u() < (0.80 if tiempo > 55 else 0.58)

    return {
        "fecha": fecha, "franja": franja, "ciudad": ciudad, "categoria": categoria,
        "n_productos": n, "valor_pedido": int(valor), "medio_pago": medio_pago,
        "estrato": estrato, "suscriptor": "Sí" if suscriptor else "No",
        "distancia_km": distancia, "lluvia": "Sí" if lluvia else "No",
        "temperatura_c": temperatura, "tiempo_entrega_min": tiempo,
        "calificacion": estrellas if califico else "",
    }


def crear_datos(carpeta="datos"):
    """Escribe <carpeta>/pedidos.csv y devuelve su ruta."""
    az = _Azar(SEMILLA)
    dias = [INICIO + timedelta(days=d) for d in range(DIAS)]
    pesos_dias = [1.35 if d.weekday() >= 4 else 1.0 for d in dias]  # viernes a domingo, más pedidos
    pedidos = [_pedido(az, dias, pesos_dias) for _ in range(N_PEDIDOS)]
    orden_franja = list(FRANJAS)
    pedidos.sort(key=lambda p: (p["fecha"], orden_franja.index(p["franja"])))

    ruta = Path(carpeta) / "pedidos.csv"
    ruta.parent.mkdir(parents=True, exist_ok=True)
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        escritor = csv.DictWriter(f, fieldnames=COLUMNAS)
        escritor.writeheader()
        for i, p in enumerate(pedidos, start=1):
            escritor.writerow({"id_pedido": f"P{i:05d}", **p, "fecha": p["fecha"].isoformat()})
    miles = f"{N_PEDIDOS:,}".replace(",", "\u202f")  # 12 000, con espacio fino
    print(f"Listo: {ruta} con {miles} pedidos y {len(COLUMNAS)} columnas")
    return ruta


"""Ayudas comunes a todos los notebooks del curso (van en la celda oculta, tras genera_datos.py).

Las de cada sesión están en ayudas_sesionN.py.
"""


def _aviso_version():
    """Las diapositivas se ejecutaron con pandas 2.2; con pandas 3 algunas salidas cambian."""
    import pandas as pd

    if int(pd.__version__.split(".")[0]) >= 3:
        print(f"Aviso: este Colab trae pandas {pd.__version__} y las diapositivas usan pandas 2.2. "
              "Algunas salidas se verán un poco distintas (por ejemplo, el texto sale como str "
              "y no como object).")


_aviso_version()


"""Sesión 6: estilo de las gráficas del curso y los verificadores de la práctica.

Van dentro de la celda oculta del notebook, junto con genera_datos.py y ayudas.py. Cada
verificador recibe lo que el estudiante calculó, dice qué está bien y, en lo demás, da una
pista sin la respuesta. Nunca se cae: un «...» sin reemplazar o un tipo inesperado se avisa.

La sesión 6 cita la verdad de la sesión 5, pero no vuelve a usar la calificación oculta: este
archivo no la reconstruye (D24 y D25).

Los valores esperados se calculan aquí, en tiempo de ejecución, desde datos/pedidos.csv.
Tolerancias: en el ejercicio 1, la media de las 1000 medias a ±0,2 min de μ y su desviación entre
1,19 y 1,34 min (con las semillas de 100 a 999, n = 100 da de 1,21 a 1,29, con o sin
replace=True; n = 120 llega hasta 1,18 y n = 80 empieza en 1,38); en el ejercicio 2, ±0,05
puntos en cada extremo (una proporción entre 0 y 1 se acepta con una pista); en el ejercicio 3,
del tamaño exacto a su redondeo hacia arriba.
"""


def _estilo_graficas():
    """Colores del curso y letras que se leen en Colab y proyectadas."""
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    plt.rcParams.update({
        "axes.prop_cycle": mpl.cycler(color=["#8A2E9E", "#F46201", "#566B7F"]),
        "figure.figsize": (6.4, 3.6),
        "figure.constrained_layout.use": True,
        "font.size": 13,
        "axes.titlesize": 15,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#94A3B8",
        "axes.labelcolor": "#334155",
        "xtick.color": "#475569",
        "ytick.color": "#475569",
        "text.color": "#1E293B",
    })


_estilo_graficas()


def _original():
    """Los pedidos tal como salen del generador, o None (con aviso) si no están."""
    import pandas as pd

    try:
        return pd.read_csv("datos/pedidos.csv")
    except FileNotFoundError:
        print("✗  No encuentro datos/pedidos.csv: ejecuta primero la celda que crea los datos "
              "(Entorno de ejecución → Ejecutar anteriores)")
        return None


def _sobrantes(sobran, otros, nombres):
    """Avisa de argumentos de más o con otro nombre. True si hubo alguno."""
    if otros:
        print(f"✗  No conozco {', '.join(repr(k) for k in otros)}: usa estos nombres, en "
              f"minúsculas y sin tildes: {', '.join(n + '=' for n in nombres)}")
        return True
    if sobran:
        print(f"✗  Sobran valores: el verificador recibe solo {len(nombres)} "
              f"({', '.join(n + '=' for n in nombres)})")
        return True
    return False


def _seguro(verificador):
    """Red de seguridad: si algo imprevisto falla al revisar, se avisa sin traceback."""
    import functools

    @functools.wraps(verificador)
    def envoltura(*args, **kwargs):
        try:
            return verificador(*args, **kwargs)
        except Exception as error:   # noqa: BLE001 — el estudiante nunca ve un traceback
            print(f"✗  No pude revisar lo que entregaste ({type(error).__name__}). Revisa que lo que "
                  "entregas sea el resultado de un cálculo de pandas y vuelve a ejecutar la celda")
    return envoltura


_COLUMNAS = {"id_pedido", "fecha", "franja", "ciudad", "categoria", "n_productos", "valor_pedido",
             "medio_pago", "estrato", "suscriptor", "distancia_km", "lluvia", "temperatura_c",
             "tiempo_entrega_min", "calificacion"}


def _un_numero(x, n_columna=None):
    """(valor, None) si x es un número, o una serie o lista con un solo número; si no, (None, motivo)."""
    import math
    import numbers
    import numpy as np
    import pandas as pd

    if isinstance(x, pd.DataFrame):
        if x.size != 1:
            return None, f"es una tabla de {x.shape[0]} × {x.shape[1]}; hace falta un solo número"
        x = x.iloc[0, 0]
    if isinstance(x, (pd.Series, pd.Index, np.ndarray, list, tuple)):
        valores = list(np.ravel(np.asarray(x, dtype=object)))
        if n_columna is not None and len(valores) == n_columna:
            return None, "esa es la columna entera, un valor por pedido: falta resumirla"
        if len(valores) == 0:
            return None, "está vacío"
        if len(valores) > 1:
            if all(isinstance(v, float) and math.isnan(v) for v in valores):
                return None, ("salió nan: si restaste dos series, pandas las alinea por su índice "
                              "y lo que no coincide queda nan; calcula cada valor como un número")
            pista = ""
            if isinstance(x, pd.Series) and set(map(str, x.index)) & _COLUMNAS:
                pista = ": ¿elegiste la columna?"   # p. ej., pedidos.quantile(0.25, numeric_only=True)
            elif isinstance(x, pd.Series) and {"count", "mean", "std"} <= set(map(str, x.index)):
                pista = ": describe() da varias medidas; quédate solo con la que se pide"
            return None, f"tiene {len(valores)} valores; hace falta uno solo{pista}"
        x = valores[0]   # quantile([0.25]) devuelve una serie de un solo valor
    if x is None:
        return None, "está vacío (None): ¿le asignaste el resultado de un print?"
    if x is pd.NA:
        return None, "salió <NA>: revisa el cálculo (¿la columna tiene datos?)"
    if callable(x):
        return None, "es una función, no su resultado: ¿faltaron los paréntesis del final, ()?"
    if isinstance(x, (bool, np.bool_)):
        return None, "es True o False; hace falta un número"
    if isinstance(x, str):
        return None, "es texto; escríbelo como número, sin comillas y con punto decimal"
    if not isinstance(x, (numbers.Real, np.number)) or isinstance(x, np.complexfloating):
        return None, f"no es un número real (es {type(x).__name__})"
    try:
        valor = float(x)
    except (TypeError, ValueError, OverflowError):
        return None, "no es un número real"
    if math.isnan(valor):
        return None, "salió nan: revisa el cálculo (¿la columna tiene datos?)"
    if math.isinf(valor):
        return None, "salió infinito: ¿dividiste entre cero?"
    return valor, None


def _cerca(a, b, tolerancia):
    return a is not None and abs(a - b) <= tolerancia


def _coma(x, decimales=2):
    """Un número con coma decimal, para los mensajes."""
    return f"{x:.{decimales}f}".replace(".", ",")


def _calificados(original):
    """Los pedidos calificados y si cada uno tiene 4 o 5 estrellas."""
    import pandas as pd

    cal = original[original["calificacion"].notna()]
    return cal, pd.to_numeric(cal["calificacion"]) >= 4


# ---------------------------------------------------------------------------------------------
# Ejercicio 1 · La distribución muestral con n = 100
# ---------------------------------------------------------------------------------------------

def _medias_como_lista(medias, n_pedidos):
    """(lista de floats, None) o (None, mensaje). Acepta lista, tupla, serie, arreglo o Index."""
    import math
    import numbers
    import numpy as np
    import pandas as pd

    x = medias
    if x is None:
        return None, "está vacía (None): ¿le asignaste el resultado de un print o de append?"
    if callable(x) and not isinstance(x, (pd.Series, pd.DataFrame)):
        return None, "es una función, no su resultado: ¿faltaron los paréntesis del final, ()?"
    if isinstance(x, str):
        return None, "es texto; hace falta la lista con las 1000 medias"
    if isinstance(x, (bool, np.bool_)):
        return None, "es True o False; hace falta la lista con las 1000 medias"
    if isinstance(x, (numbers.Number, np.number)) or (isinstance(x, np.ndarray) and x.ndim == 0):
        return None, ("es un solo número; hace falta la lista con las 1000 medias, una por "
                      "muestra")
    if isinstance(x, pd.DataFrame):
        if x.shape[1] == 1:
            x = x.iloc[:, 0]
        else:
            return None, (f"es una tabla de {x.shape[0]} × {x.shape[1]}; hace falta la lista con "
                          "las 1000 medias")
    if isinstance(x, dict):
        x = list(x.values())
    if isinstance(x, (set, frozenset)):
        return None, ("es un conjunto (set), que borra las medias repetidas: usa una lista, "
                      "medias = []")
    if not isinstance(x, (list, tuple, pd.Series, np.ndarray, pd.Index)):
        return None, f"tiene que ser una lista de números (es {type(x).__name__})"
    if isinstance(x, np.ndarray) and x.ndim > 1:
        if min(x.shape) > 1:   # una tabla de verdad, no una sola fila o columna
            return None, (f"es una tabla de {' × '.join(map(str, x.shape))}: cada fila es una "
                          "muestra entera; a medias.append le falta calcular la media, .mean()")
        x = x.ravel()
    elementos = list(x)
    if len(elementos) == 0:
        return None, ("está vacía: ¿medias.append(...) quedó fuera del for? Va con sangría, "
                      "dentro del bucle")
    if isinstance(x, pd.Series) and len(elementos) == n_pedidos:
        return None, ("esa es la columna entera, un valor por pedido: hacen falta las medias de "
                      "las 1000 muestras")
    valores = []
    for e in elementos:
        if isinstance(e, pd.DataFrame):
            return None, ("cada elemento es una muestra entera (una tabla de pedidos): elige la "
                          "columna del tiempo y calcula su media")
        if isinstance(e, (pd.Series, np.ndarray, list, tuple)):
            partes = list(np.ravel(np.asarray(e, dtype=object)))
            if len(partes) > 1 and isinstance(e, pd.Series) and set(map(str, e.index)) & _COLUMNAS:
                return None, ("cada elemento tiene un valor por columna: calculaste la media de "
                              "todas las columnas; quédate con la de tiempo_entrega_min")
            if len(partes) == 1:
                e = partes[0]
            elif len(partes) == 0:
                return None, ("cada elemento es una muestra vacía: ¿n=0? Saca muestras del "
                              "tamaño que pide el ejercicio")
            else:
                return None, (f"cada elemento es una muestra entera ({len(partes)} tiempos), no "
                              "su media: a medias.append le falta calcular la media, .mean()")
        if e is Ellipsis:
            return None, "todavía tiene ...: completa la línea de medias.append"
        if e is None:
            return None, ("tiene valores vacíos (None): ¿guardaste el resultado de un print? "
                          "Guarda la media")
        if isinstance(e, (bool, np.bool_)):
            return None, "tiene True o False; hacen falta las medias del tiempo"
        if isinstance(e, str):
            return None, "tiene texto; hacen falta números: las medias del tiempo"
        if callable(e):
            return None, ("tiene funciones, no resultados: ¿faltaron los paréntesis de "
                          ".mean()?")
        if not isinstance(e, (numbers.Real, np.number)) or isinstance(e, np.complexfloating):
            return None, f"tiene valores que no son números ({type(e).__name__})"
        try:
            v = float(e)
        except (TypeError, ValueError, OverflowError):
            return None, "tiene valores que no son números"
        if math.isnan(v):
            return None, ("tiene medias nan: ¿alguna muestra quedó vacía (n=0) o elegiste una "
                          "columna sin datos?")
        if math.isinf(v):
            return None, "tiene medias infinitas: revisa el cálculo"
        valores.append(v)
    return valores, None


_REFERENCIAS = {}


def _medias_de_referencia(original):
    """Las medias de n = 100 con mi_semilla = 123 y con las semillas de la clase (se guardan)."""
    import numpy as np

    t = original["tiempo_entrega_min"]
    clave = (len(t), float(t.sum()))
    if clave not in _REFERENCIAS:
        _REFERENCIAS[clave] = (
            np.array([t.sample(n=100, random_state=123 + i).mean() for i in range(1000)]),
            np.array([t.sample(n=100, random_state=i).mean() for i in range(1000)]))
    return _REFERENCIAS[clave]


@_seguro
def revisar_medias(medias=..., *sobran, **otros):
    """Ejercicio 1: las medias de 1000 muestras de 100 pedidos."""
    import numpy as np

    if _sobrantes(sobran, otros, ["medias"]):
        return
    if medias is Ellipsis:
        print("·  medias: todavía tiene ...")
        return
    original = _original()
    if original is None:
        return
    tiempo = original["tiempo_entrega_min"]
    mu = float(tiempo.mean())
    sigma = float(tiempo.std(ddof=0))
    valores, problema = _medias_como_lista(medias, len(original))
    if problema:
        print("✗  medias: " + problema)
        return
    k = len(valores)
    if k == 1:
        print("✗  medias: tiene una sola media: ¿medias.append quedó fuera del for, o medias = [] "
              "quedó dentro? El append va con sangría, dentro del bucle; medias = [], antes")
        return
    if k != 1000:
        if k % 1000 == 0:
            print(f"✗  medias: tiene {k} medias: ¿ejecutaste el bucle {k // 1000} veces sin volver a "
                  "crear la lista? Ejecuta la celda completa, con medias = [] arriba")
        else:
            print(f"✗  medias: tiene {k} medias; se piden 1000, una por muestra: for i in range(1000)")
        return
    v = np.asarray(valores, dtype=float)
    centro, desv = float(v.mean()), float(v.std(ddof=1))
    if desv < 1e-9:
        if abs(centro - mu) < 1e-6:
            print("✗  medias: todas valen μ: cada «muestra» es la tabla entera. Saca muestras de "
                  "100 pedidos")
        elif centro >= 1 and float(centro).is_integer():
            print(f"✗  medias: todas valen {centro:.0f}: eso es cuántos pedidos trae cada muestra "
                  "(.count(), len o .size), no su media. Usa .mean()")
        else:
            print("✗  medias: todas las medias son iguales: las 1000 muestras salieron idénticas. "
                  "¿Dejaste random_state fijo? Usa random_state=mi_semilla + i, que cambia en "
                  "cada vuelta")
        return
    lejos = abs(centro - mu)
    if lejos <= 0.2 and 1.19 <= desv <= 1.34:
        print("✓  medias: 1000 medias de muestras de 100 pedidos")
        print(f"Tus medias: centro {_coma(centro)} min (μ = {_coma(mu)} min) · EE = "
              f"{_coma(desv)} min")
        print(f"La fórmula: σ / √100 = {_coma(sigma)} / 10 = {_coma(sigma / 10)} min")
        de_123, de_clase = _medias_de_referencia(original)
        if np.allclose(v, de_123):
            print("·  Son las medias de mi_semilla = 123: cámbiala por un número de tres cifras "
                  "que elijas tú y vuelve a ejecutar")
        elif np.allclose(v, de_clase):
            print("·  Son las semillas de la clase (random_state=i): usa random_state=mi_semilla "
                  "+ i, con un número de tres cifras que elijas tú")
        print("✓  Cuadra. Responde abajo: con n = 400, ¿cuánto quedaría el EE, y por qué?")
        return
    if 33.4 <= centro <= 34.1 and 0.9 <= desv <= 1.6:
        print("✗  medias: esas son medianas, no medias: el centro queda en unos 33,8 min, el de la "
              "mediana del tiempo. Usa .mean()")
    elif 10.5 <= centro <= 13.5 and desv < 6:
        print("✗  medias: el centro de tus valores es de unos 12 min, la desviación estándar de los "
              "pedidos: ¿usaste .std() en vez de .mean()?")
    elif any(abs(centro / mu - k) <= 0.1 * k for k in (10, 25, 80, 100, 120, 400, 1000)):
        print("✗  medias: tus valores son muchas veces μ: ¿sumaste los tiempos (.sum()) en vez de "
              "promediarlos (.mean())?")
    elif lejos > 1.5:
        print(f"✗  medias: el centro de tus medias es {_coma(centro)}, lejos de los "
              f"{_coma(mu)} min del tiempo de entrega: ¿elegiste otra columna, filtraste los "
              "pedidos o usaste otra cosa en vez de .mean()? Va la media de tiempo_entrega_min, "
              "con muestras de todos los pedidos")
    elif 9 <= desv <= 14:
        print("✗  medias: tus medias varían tanto como los pedidos sueltos: cada muestra trae un "
              "solo pedido (¿n=1?). Saca muestras de 100 pedidos")
    elif lejos > 0.25:
        print(f"✗  medias: el centro de tus medias, {_coma(centro)} min, se corrió de μ "
              f"({_coma(mu)}) más de lo que da el azar: ¿filtraste los pedidos antes de sortear? "
              "Las muestras salen de todos los pedidos: pedidos.sample(...)")
    elif 0.59 <= desv <= 0.66:
        print("✗  medias: varían como las medias de 400 pedidos, el tamaño de la clase: cambia el "
              "400 por el tamaño que pide el ejercicio")
    elif 2.45 <= desv <= 2.72:
        print("✗  medias: varían como las medias de 25 pedidos: tus muestras son más chicas de lo "
              "que pide el ejercicio")
    elif 3.5 <= desv <= 4.25:
        print("✗  medias: varían como las medias de 10 pedidos: tus muestras son más chicas de lo "
              "que pide el ejercicio")
    elif 1.34 < desv <= 1.5:
        print("✗  medias: varían un poco más de lo esperado: tus muestras son algo más chicas que "
              "las que pide el ejercicio (¿n=80?)")
    elif 1.08 <= desv < 1.19:
        print("✗  medias: varían un poco menos de lo esperado: tus muestras son algo más grandes "
              "(¿n=120 o frac=0.01?). frac=0.01 es el 1 % de 12 000, o sea 120 pedidos; el "
              "ejercicio pide un número de pedidos, n=")
    elif desv < 1.19:
        print("✗  medias: varían menos de lo esperado: tus muestras son más grandes que las que "
              "pide el ejercicio. Revisa el n=")
    elif desv > 1.34:
        print("✗  medias: varían más de lo esperado: tus muestras son más chicas que las que pide "
              "el ejercicio. Revisa el n=")
    else:
        print("✗  medias: el centro se corrió de μ más de lo que da el azar: revisa que cada muestra "
              "salga de todos los pedidos (pedidos.sample) y que guardes la media del tiempo")


# ---------------------------------------------------------------------------------------------
# Ejercicio 2 · El intervalo de la semana 20
# ---------------------------------------------------------------------------------------------

def _intervalos_semana(original):
    """El intervalo correcto de la semana 20 y los de los errores probables, en %."""
    import math
    import pandas as pd

    fecha = pd.to_datetime(original["fecha"])
    semana = fecha.dt.isocalendar().week.astype(int)
    cal, buena = _calificados(original)
    p_sem = float(buena.mean())

    def ic(p, n, z=1.96, ee=None):
        e = math.sqrt(p * (1 - p) / n) if ee is None else ee
        return ((p - z * e) * 100, (p + z * e) * 100)

    w = original[semana == 20]
    c = w[w["calificacion"].notna()]
    estrellas = pd.to_numeric(c["calificacion"])
    p, n = float((estrellas >= 4).mean()), len(c)
    todos = float((pd.to_numeric(w["calificacion"]) >= 4).mean())   # el faltante cuenta como False
    w21 = original[semana == 21]
    c21 = pd.to_numeric(w21["calificacion"].dropna())
    e = math.sqrt(p * (1 - p) / n)
    correcto = ic(p, n)
    pistas = [
        (ic(todos, len(w)), "calculaste p̂ sobre todos los pedidos de la semana, y los que no "
                            "calificaron cuentan como «no 4 o 5»: p̂ y n son de los calificados"),
        (ic(p, len(w)), "usaste como n todos los pedidos de la semana; n son los calificados"),
        (ic(p, len(cal)), "usaste como n todos los calificados del semestre (la variable n de "
                          "[proporcion]); n son los calificados de la semana 20"),
        (ic(p, len(original)), "usaste como n los 12 000 pedidos; n son los calificados de la "
                               "semana 20"),
        (ic(p, n, 1.645), "ese es el intervalo de 90 % (z = 1,645); el de 95 % usa 1,96"),
        (ic(p, n, 2.576), "ese es el intervalo de 99 % (z = 2,576); el de 95 % usa 1,96"),
        (ic(p, n, 2), "usaste z = 2; el intervalo de 95 % usa 1,96"),
        (ic(p, n, ee=math.sqrt(p_sem * (1 - p_sem) / n)),
         "en el EE usaste la p̂ del semestre; va la de la semana 20"),
        (ic(float((estrellas >= 3).mean()), n), "contaste 3, 4 o 5 estrellas (>= 3); se pide 4 o 5"),
        (ic(float((estrellas == 5).mean()), n), "contaste solo las de 5 estrellas; se pide 4 o 5 "
                                                "(>= 4)"),
        (ic(1 - p, n), "ese es el intervalo de 1 − p̂, los que NO tienen 4 o 5 estrellas"),
        (ic(p, n, 1), "sumaste y restaste un solo EE; el margen es 1,96 · EE"),
        (ic(float((c21 >= 4).mean()), len(c21)),
         "esa es la semana ISO 21: ¿usaste %W o strftime? La semana va con "
         ".dt.isocalendar().week, como en [semanal]"),
    ]
    angostos = [((p - 1.96 * p * (1 - p) / n) * 100, (p + 1.96 * p * (1 - p) / n) * 100),
                (p * 100 - 1.96 * e, p * 100 + 1.96 * e)]
    return {"correcto": correcto, "pistas": pistas, "angostos": angostos, "p": p * 100, "n": n,
            "p_sem": p_sem * 100}


@_seguro
def revisar_intervalo(*sueltos, inferior=..., superior=..., **otros):
    """Ejercicio 2: el intervalo de 95 % del % de 4 o 5 estrellas de la semana 20, en %."""
    import pandas as pd

    nombres = ["inferior", "superior"]
    if sueltos:
        print("✗  Pasa los valores por nombre, como en la plantilla: "
              "revisar_intervalo(inferior=inferior, superior=superior)")
        return
    if _sobrantes((), otros, nombres):
        return
    original = _original()
    if original is None:
        return
    r = _intervalos_semana(original)
    lo, hi = r["correcto"]
    tol, tol_pista = 0.05, 0.06
    vals = {}
    for nombre, x in [("inferior", inferior), ("superior", superior)]:
        if x is Ellipsis:
            print(f"·  {nombre}: todavía tiene ...")
            continue
        val, problema = _un_numero(x, len(original))
        if problema:
            if isinstance(x, (pd.Series, pd.DataFrame)) and len(x) > 1:
                filas = f"{len(x):,}".replace(",", "\u202f")   # 12 000, con espacio fino
                problema = (f"es una columna o una tabla, con {filas} filas: hace falta un "
                            "solo número, el extremo del intervalo")
            elif problema.startswith("salió nan"):
                problema = ("salió nan: ¿la raíz de un número negativo (si p̂ va en %, el EE usa "
                            "100 − p̂, no 1 − p̂) o una semana sin calificados (revisa el filtro "
                            "semana == 20)?")
            print(f"✗  {nombre}: {problema}")
            continue
        vals[nombre] = val
    if len(vals) < 2:
        return
    a, b = vals["inferior"], vals["superior"]

    def es(par, t, escala=1.0):
        return _cerca(a, par[0] * escala, t * escala) and _cerca(b, par[1] * escala, t * escala)

    if es((lo, hi), tol):
        print("✓  inferior y superior: el intervalo de 95 % de la semana 20")
    elif es((lo, hi), tol, 0.01):
        print("✓  inferior y superior (como proporción)")
        print("·  Diste proporciones (entre 0 y 1); el ejercicio lo pide en %: multiplica por 100")
    elif es((hi, lo), tol) or es((hi, lo), tol, 0.01):
        print("✗  ¿Intercambiaste los dos? inferior es el extremo menor (p̂ − margen) y superior, "
              "el mayor")
        return
    else:
        for par, pista in r["pistas"]:
            if es(par, tol_pista) or es(par, tol_pista, 0.01) or \
                    es((par[1], par[0]), tol_pista) or es((par[1], par[0]), tol_pista, 0.01):
                print(f"✗  {pista}")
                return
        def redondeado(x, ref, dec):
            return round(x, dec) == x and abs(x - ref) <= 0.5 * 10 ** -dec + 1e-9

        if (redondeado(a, lo, 0) and redondeado(b, hi, 0)) or \
                (redondeado(a, lo / 100, 2) and redondeado(b, hi / 100, 2)):
            print("✗  casi: redondeaste demasiado. Entrega los extremos sin redondear, o en % con "
                  "al menos un decimal")
            return
        ancho = b - a
        escala = 0.01 if max(abs(a), abs(b)) <= 1.0 else 1.0
        centro = (a + b) / 2 / escala
        if a == b:
            print("✗  inferior y superior son iguales: falta sumar y restar el margen, 1,96 · EE")
        elif min(a, b) / escala < 0 or max(a, b) / escala > 100:
            print("✗  un porcentaje va de 0 a 100: revisa el cálculo (¿multiplicaste por 100 dos "
                  "veces, o sumaste el margen en otra escala?)")
        elif any(es(par, tol_pista) or es(par, tol_pista, 0.01) for par in r["angostos"]) or \
                (0 < abs(ancho) / escala < 1 and abs(centro - r["p"]) < 0.5):
            print("✗  tu intervalo mide menos de un punto de ancho: revisa la raíz de la fórmula "
                  "y que el margen esté en la misma escala que p̂ (las dos en %, o las dos en "
                  "proporción)")
        elif 1 <= min(a, b) and max(a, b) <= 5:
            print("✗  eso parece un intervalo para la media de las estrellas: la satisfacción no se "
                  "promedia; cuenta qué parte de los calificados tiene 4 o 5 estrellas, en %")
        elif a > b:
            print("✗  inferior es mayor que superior: inferior es p̂ − margen y superior, "
                  "p̂ + margen")
        elif abs(centro - r["p"]) <= 0.3:
            print("✗  el centro está bien (la p̂ de la semana 20), pero el margen no: es 1,96 · EE, "
                  "con EE = √(p̂ (1 − p̂) / n)")
        elif abs(centro - r["p_sem"]) <= 0.3:
            print("✗  ese intervalo está centrado en la cifra del semestre; se pide el de la semana "
                  "20: filtra semana == 20")
        else:
            print("✗  no cuadra: revisa que uses los calificados de la semana 20 y el % de 4 o 5 "
                  "estrellas")
        return
    print(f"Semana 20: p̂ = {_coma(r['p'])} % con n = {r['n']} calificados · intervalo "
          f"[{_coma(lo)}; {_coma(hi)}] % · semestre: {_coma(r['p_sem'])} %")
    print("✓  Cuadra. Responde abajo: ¿fue la semana 20 peor que el semestre?")


# ---------------------------------------------------------------------------------------------
# Ejercicio 3 (opcional) · ¿Cuántos calificados necesita Bucaramanga para ±5 puntos?
# ---------------------------------------------------------------------------------------------

@_seguro
def revisar_tamano(*sueltos, n=..., **otros):
    """Ejercicio 3: el tamaño de muestra de Bucaramanga para un margen de ±5 puntos."""
    import math
    import pandas as pd

    if sueltos:
        print("✗  Pasa el valor por nombre, como en la plantilla: revisar_tamano(n=n_necesario)")
        return
    if _sobrantes((), otros, ["n"]):
        return
    if n is Ellipsis:
        print("·  n: todavía tiene ...")
        return
    original = _original()
    if original is None:
        return
    val, problema = _un_numero(n, len(original))
    if problema:
        if isinstance(n, (pd.Series, pd.DataFrame)) and len(n) > 1:
            problema = "es una columna o una tabla; hace falta un solo número, el tamaño"
        print(f"✗  n: {problema}")
        return
    cal, buena = _calificados(original)
    en_buc = cal["ciudad"] == "Bucaramanga"
    p = float(buena[en_buc].mean())
    p_todos = float((pd.to_numeric(original["calificacion"]) >= 4)[
        original["ciudad"] == "Bucaramanga"].mean())
    p_nac = float(buena.mean())
    z2 = 1.96 ** 2

    def tam(q, e=0.05, z=1.96):
        return z ** 2 * q * (1 - q) / e ** 2

    exacto = tam(p)
    prudente = tam(0.5)

    def cerca_de(x):   # el valor exacto o redondeado (los muy pequeños, solo el exacto)
        if x < 1:
            return abs(val - x) <= 0.05 * abs(x)
        return x - 0.02 - 1e-9 <= val <= math.ceil(x) + 1e-9 or abs(val - x) <= 0.002 * abs(x)

    if val <= 0:
        print("✗  n: un tamaño de muestra es un número positivo: revisa el cálculo (si p̂ va "
              "en %, el complemento es 100 − p̂, no 1 − p̂)")
        return

    if exacto - 0.02 <= val <= math.ceil(exacto) + 1e-9:
        print("✓  n: el tamaño para ±5 puntos en Bucaramanga")
        if val < math.ceil(exacto):
            print("·  Un tamaño de muestra se redondea siempre hacia arriba: con uno menos, el "
                  "margen pasaría de ±5")
        mediana = cal[en_buc].groupby(
            pd.to_datetime(cal.loc[en_buc, "fecha"]).dt.isocalendar().week).size().median()
        print(f"Bucaramanga: p̂ = {_coma(p * 100, 1)} % de 4 o 5 estrellas · unos {mediana:.0f} "
              "calificados por semana")
        print("✓  Cuadra. Responde abajo: ¿cuántas semanas de Bucaramanga son? ¿Por semana o por "
              "mes?")
        return
    if prudente - 0.06 <= val <= math.ceil(prudente) + 1e-9:   # de 384,1 a 385
        print("✓  n: con p = 0,5, el tamaño más prudente: sirve para cualquier p̂")
        print("·  Es la opción prudente cuando no se conoce p̂. Con la p̂ de Bucaramanga, que ya "
              "conoces, hacen falta bastantes menos: pruébalo y compara")
        return
    if math.floor(exacto) <= val < exacto - 0.02:
        abajo = math.floor(val)
        margen = 1.96 * math.sqrt(p * (1 - p) / abajo) * 100
        print(f"✗  n: con {abajo} calificados el margen sería de ±{_coma(margen)} puntos, más de "
              "±5: un tamaño de muestra se redondea siempre hacia arriba")
        return
    if math.floor(prudente) <= val < prudente - 0.06:
        print("✗  n: ese es el tamaño con p = 0,5, pero redondeado hacia abajo: un tamaño de "
              "muestra se redondea siempre hacia arriba (con uno menos, el margen pasa de ±5)")
        return
    pistas = [
        (tam(p_todos), "calculaste p̂ sobre todos los pedidos de Bucaramanga, y los que no "
                       "calificaron cuentan como «no 4 o 5»: p̂ es entre los calificados"),
        (tam(p_nac), "esa es la p̂ nacional; se pide la de Bucaramanga"),
        (z2 * p * (1 - p) / 5 ** 2, "con p̂ en proporción, E también va en proporción: 0.05, no 5"),
        (z2 * (p * 100) * (100 - p * 100) / 0.05 ** 2, "con p̂ en %, E también va en %: 5, no 0.05"),
        (1.96 * p * (1 - p) / 0.05 ** 2, "el 1,96 va al cuadrado: 1.96 ** 2"),
        (z2 * p * (1 - p) / 0.05, "E va al cuadrado: 0.05 ** 2"),
        (tam(p, 0.025), "usaste la mitad de E: el margen es lo que se suma y se resta (±5), no el "
                        "ancho del intervalo"),
        (tam(p, 0.05, 2), "usaste z = 2; el 95 % usa 1,96"),
        (tam(p, 0.05, 1.645), "usaste z = 1,645, que es el de 90 %; el de 95 % usa 1,96"),
        (tam(p, 0.05, 2.576), "usaste z = 2,576, que es el de 99 %; el de 95 % usa 1,96"),
        (math.sqrt(exacto), "sacaste la raíz: la fórmula del tamaño no lleva raíz"),
    ]
    for x, pista in pistas:
        if cerca_de(x):
            print(f"✗  n: {pista}")
            return
    print("✗  n: no cuadra: revisa p̂ (4 o 5 estrellas entre los calificados de Bucaramanga) "
          "y E (0.05 si p̂ va en proporción)")

# Crea datos/pedidos.csv: los mismos 12 000 pedidos de las sesiones 1 a 5
ruta = crear_datos("datos")

## 1 · Los datos, con sus escalas

Cargamos los pedidos y declaramos las escalas, como en las sesiones anteriores. También usamos
`numpy`, que se importa como `np`.

In [ ]:
# [cargar]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pedidos = pd.read_csv("datos/pedidos.csv")
# Las escalas que declaramos en la sesión 1
franjas = ["Mañana", "Almuerzo", "Tarde", "Noche"]
pedidos["franja"] = pd.Categorical(
    pedidos["franja"], categories=franjas, ordered=True)
pedidos["estrato"] = pd.Categorical(
    pedidos["estrato"], categories=[1, 2, 3, 4, 5, 6], ordered=True)
pedidos["calificacion"] = pd.Categorical(
    pedidos["calificacion"], categories=[1, 2, 3, 4, 5], ordered=True)
pedidos.shape

### Diccionario de datos (el mismo de la sesión 1)

| Columna | Qué registra | Valores o unidad |
|---|---|---|
| `id_pedido` | Código del pedido | P00001, P00002… |
| `fecha` | Día del pedido | del 1 de enero al 30 de junio de 2026 |
| `franja` | Momento del día en que se pidió | Mañana, Almuerzo, Tarde, Noche |
| `ciudad` | Ciudad de entrega | Bogotá, Medellín, Cali, Barranquilla, Bucaramanga |
| `categoria` | Tipo de comercio que despacha | Restaurante, Mercado, Farmacia, Mascotas, Floristería |
| `n_productos` | Unidades pedidas | conteo |
| `valor_pedido` | Valor de los productos, sin domicilio ni propina | pesos (COP) |
| `medio_pago` | Cómo se pagó | Efectivo, Tarjeta, Billetera digital |
| `estrato` | Estrato de la **dirección de entrega** (lo asigna el municipio a la vivienda) | 1 a 6 |
| `suscriptor` | Si quien pide paga la suscripción de Tingua | Sí / No |
| `distancia_km` | Distancia por vía del comercio a la dirección | kilómetros; la app registra hasta 14: una ruta más larga queda como 14 |
| `lluvia` | Si llovía en la zona durante la entrega | Sí / No |
| `temperatura_c` | Temperatura en la zona al momento de la entrega | grados Celsius |
| `tiempo_entrega_min` | Minutos desde que se confirma el pedido hasta que llega | minutos |
| `calificacion` | Estrellas que dio quien pidió; vacía si no calificó | 1 a 5 |

## La pregunta de hoy: ¿cuánto confiar en una muestra?

En la sesión 5, la cifra de satisfacción del tablero fue el 79,1 % de 4 o 5 estrellas, con su
«¿de quiénes?»: los 7099 pedidos que se calificaron. Hoy la pregunta es otra. Una cifra que sale de
una muestra es una **estimación puntual**: con otra muestra habría salido otra. ¿Cuánto puede
alejarse de la verdad solo por azar? Eso lo dice su **margen**.

En la sesión 3 el lema fue «Ningún centro sin su dispersión»: era la dispersión de los pedidos. Hoy
es la de las cifras que salen de una muestra: **ninguna cifra sin su margen**.

Primero veremos qué pasa con muchas muestras (la distribución muestral); después, cómo poner el
margen cuando hay una sola (el intervalo de confianza).

La pregunta de entrada de la clase: en la semana 13, el 86,2 % de los pedidos calificados tuvo 4 o 5
estrellas, la mejor del semestre. ¿Premiamos al equipo? La respondemos en la sección 4.

## 2 · ¿Cuánto puede alejarse una muestra?

En la sesión 5, las cuatro muestras al azar de 400 pedidos cayeron cerca de $\mu$. ¿Fue suerte?
Repetimos el sorteo 1000 veces, cada vez con otra semilla, y guardamos la media de cada muestra.
Como conocemos los 12 000 pedidos, sabemos dónde está $\mu$: el blanco.

In [ ]:
# [mil_muestras]
tiempo = pedidos["tiempo_entrega_min"]
mu = tiempo.mean()                         # el parámetro: lo conocemos
medias = []
for i in range(1000):                      # 1000 muestras de 400 pedidos
    muestra = pedidos.sample(n=400, random_state=i)
    medias.append(muestra["tiempo_entrega_min"].mean())
medias = pd.Series(medias)
print(f"μ = {mu:.2f} min")
print(f"Las 1000 medias: de {medias.min():.2f} a {medias.max():.2f} min,",
      f"con media {medias.mean():.2f}")

Las 1000 medias forman la **distribución muestral** de $\bar x$: cómo se reparte la media de una
muestra de 400 pedidos entre muchas muestras. A la izquierda, los pedidos; a la derecha, las medias,
con una curva normal encima (centrada en $\mu$ y con la misma desviación que las medias).

In [ ]:
# [mil_muestras_grafica]
from statistics import NormalDist
fig, (izq, der) = plt.subplots(1, 2, figsize=(9.6, 3.8))
izq.hist(tiempo, bins=range(0, 205, 5), color="#8A2E9E")
izq.set_title("Los 12 000 pedidos")
izq.set_xlabel("Tiempo de entrega (min)")
izq.set_ylabel("Pedidos")
der.hist(medias, bins=30, density=True, color="#8A2E9E", alpha=0.55)
normal = NormalDist(mu, medias.std())   # mismo centro, misma desviación
x = np.linspace(medias.min(), medias.max(), 200)
der.plot(x, [normal.pdf(v) for v in x], color="#F46201", linewidth=2.5)
der.set_yticks([])                         # importa la forma, no la altura
der.set_title("Las 1000 medias de 400 pedidos")
der.set_xlabel("Media del tiempo (min)")
titulo = "Los pedidos son asimétricos; las medias de 400, casi normales"
fig.suptitle(titulo, x=0.01, ha="left", fontsize=15, fontweight="bold")
plt.show()

Los pedidos tienen una cola larga a la derecha, hasta 200,9 min. Las medias de 400 no: forman una
campana estrecha alrededor de $\mu$, y la curva normal las describe bien. Fíjate en los ejes: los
pedidos van de 0 a 200 min; las medias, de 33 a 38.

### El error estándar

- La **desviación estándar** de los pedidos (sesión 3) dice cuánto varía **un pedido**.
- La desviación de las 1000 medias dice cuánto varía **la media de una muestra**: se llama **error
  estándar** (EE). Describe una cifra, no los pedidos.

No hace falta simular para conocerlo. Si $\sigma$ es la desviación de los pedidos y $n$ el tamaño de
la muestra,

$$\mathrm{EE} = \frac{\sigma}{\sqrt{n}}$$

In [ ]:
# [error_estandar]
sigma = tiempo.std(ddof=0)                 # la de los 12 000 pedidos
ee = medias.std()                          # la de las 1000 medias
print(f"Desviación estándar de los pedidos:  σ  = {sigma:.2f} min")
print(f"Error estándar de las medias de 400: EE = {ee:.2f} min")
print(f"La fórmula: σ / √400 = {sigma / np.sqrt(400):.2f} min")

La simulación (0,62) y la fórmula (0,63) casi coinciden. **Antes de seguir, predice:** si la gerencia
toma 1600 pedidos en vez de 400, ¿el EE baja a la mitad o a la cuarta parte? La celda siguiente
repite la simulación con 25, 100 y 400 pedidos: cada tamaño es 4 veces el anterior.

In [ ]:
# [tres_n]
bordes = np.arange(28, 48, 0.4)           # las mismas barras en los tres
fig, ejes = plt.subplots(1, 3, figsize=(9.6, 3.4), sharex=True)
for n, ax in zip([25, 100, 400], ejes):
    medias_n = pd.Series([tiempo.sample(n=n, random_state=i).mean()
                          for i in range(1000)])
    print(f"n = {n:3}: EE = {medias_n.std():.2f}",
          f"(fórmula {sigma / np.sqrt(n):.2f})",
          f"· asimetría {medias_n.skew():.2f}")
    ax.hist(medias_n, bins=bordes, color="#8A2E9E")
    ax.set_title(f"n = {n} · EE = {medias_n.std():.2f}")
    ax.set_yticks([])
ejes[1].set_xlabel("Media del tiempo de entrega (min)")
fig.suptitle("Con 4 veces más pedidos, el error estándar baja a la mitad",
             x=0.01, ha="left", fontsize=15, fontweight="bold")
plt.show()

Cuatro veces más pedidos dan la mitad de EE, porque $\sqrt 4 = 2$. Con $n = 25$ las medias todavía se
inclinan un poco a la derecha (su asimetría es la más alta); con 400, casi nada.

### La distribución normal

Es lo que prometieron las sesiones 3 y 4. El **teorema del límite central** dice que, con $n$ grande,
las medias de las muestras se reparten casi como una **normal**, aunque los datos no lo sean. En una
normal, el 68 % queda a menos de una desviación del centro y el 95 % a menos de 1,96 desviaciones.

- En las medias de 400 se cumple casi exacto: es una normal con desviación EE.
- En los pedidos, no: a una desviación de la media hay un 81,7 % (sesión 3), porque no son normales.

El puntaje $z$ de la sesión 4 también vale para una cifra: $z = (\bar x - \mu)/\mathrm{EE}$ dice a
cuántos errores estándar de $\mu$ cayó una media. En el 95 % de las muestras, $|z| \le 1{,}96$.

### ¿Qué método de muestreo es más preciso?

La sesión 5 lo dejó pendiente: con una sola muestra por método no se podía saber. Ahora repetimos
1000 veces cada método al azar (el sistemático solo tiene 30 inicios posibles) y medimos el error
estándar de cada uno. Las medias de la aleatoria simple ya las tenemos: son `medias`. Tarda unos
segundos.

In [ ]:
# [metodos_precision]
fechas = pedidos["fecha"].drop_duplicates()        # los 181 días
metodos = {"Aleatoria simple": list(medias), "Sistemática": [],
           "Estratificada": [], "Por conglomerados": []}
for i in range(1000):
    estratificada = pedidos.groupby("ciudad").sample(frac=1/30,
                                                     random_state=i)
    metodos["Estratificada"].append(
        estratificada["tiempo_entrega_min"].mean())
    dias = fechas.sample(n=6, random_state=i)
    de_los_dias = pedidos[pedidos["fecha"].isin(dias)]
    metodos["Por conglomerados"].append(
        de_los_dias["tiempo_entrega_min"].mean())
for inicio in range(30):                           # los 30 inicios posibles
    metodos["Sistemática"].append(tiempo.iloc[inicio::30].mean())
ee_metodo = pd.Series({m: np.std(v) for m, v in metodos.items()})
print("Error estándar de cada método (min):")
print(ee_metodo.round(2).to_string())

In [ ]:
# [metodos_precision_grafica]
conveniencia = pedidos[pedidos["ciudad"] == "Bogotá"].head(400)
x_conv = conveniencia["tiempo_entrega_min"].mean()   # no depende del azar
desorden = np.random.default_rng(0)   # separa los puntos de cada franja
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for fila, (m, v) in enumerate(metodos.items()):
    y = fila + desorden.uniform(-0.25, 0.25, len(v))
    ax.scatter(v, y, s=8, alpha=0.3, color="#8A2E9E")
    ax.text(max(v) + 0.2, fila, f"EE = {np.std(v):.2f}", va="center")
ax.scatter(x_conv, len(metodos), s=110, color="#F46201", zorder=3)
ax.text(x_conv - 0.3, len(metodos), f"siempre {x_conv:.2f}", va="center",
        ha="right")
ax.axvline(mu, color="#3A0F47", linewidth=2, linestyle="--")
ax.set_yticks(range(len(metodos) + 1), list(metodos) + ["Por conveniencia"])
ax.set_ylim(len(metodos) + 0.6, -0.6)              # en el orden de la lista
ax.set_title("El azar da el ancho; el sesgo mueve el centro")
ax.set_xlabel("Media del tiempo de entrega (min); la línea es μ")
plt.show()

En Tingua, los cuatro métodos al azar tienen casi la misma precisión: sus franjas son casi igual de
anchas y están centradas en $\mu$. Las centésimas de diferencia son en parte ruido de la simulación.
La muestra por conveniencia cae siempre en el mismo punto, lejos de $\mu$: el azar da el ancho; el
sesgo mueve el centro.

## 3 · En la vida real hay una sola muestra

No conocemos $\mu$ ni $\sigma$: tenemos una muestra. En la fórmula del EE, $\sigma$ se reemplaza por la
desviación de la muestra, $s$ (la que divide entre $n - 1$, sesión 3):

$$\mathrm{EE} \approx \frac{s}{\sqrt{n}}$$

Usamos la muestra aleatoria simple de la sesión 5, la de la semilla 1.

In [ ]:
# [una_muestra]
muestra = pedidos.sample(n=400, random_state=1)   # la de la sesión 5
x_barra = muestra["tiempo_entrega_min"].mean()
s = muestra["tiempo_entrega_min"].std()           # divide entre n − 1
ee_1 = s / np.sqrt(400)
print(f"x̄ = {x_barra:.2f} min · s = {s:.2f} min · EE = {ee_1:.3f} min")

### El intervalo de confianza de 95 %

En el 95 % de las muestras, $\bar x$ cae a menos de 1,96 EE de $\mu$. Dándole la vuelta: si a cada
$\bar x$ le sumamos y restamos 1,96 EE, el 95 % de esos intervalos atrapa a $\mu$. El **margen** es
$1{,}96 \cdot \mathrm{EE}$ (en las encuestas lo llaman «margen de error»):

$$\bar x \pm 1{,}96\,\mathrm{EE}$$

In [ ]:
# [intervalo]
margen = 1.96 * ee_1          # el 1,96 deja el 95 % central de la normal
inferior, superior = x_barra - margen, x_barra + margen
print(f"Margen: ±{margen:.2f} min")
print(f"Intervalo de 95 %: [{inferior:.2f}; {superior:.2f}] min")
print(f"¿Contiene a μ = {mu:.2f}? {inferior <= mu <= superior}")

En la vida real no sabríamos si este intervalo contiene a $\mu$. ¿Qué quiere decir entonces el 95 %?
Repetimos el intervalo con 100 muestras, las de las semillas 1 a 100.

In [ ]:
# [cien_intervalos]
filas = []
for semilla in range(1, 101):             # 100 muestras, 100 intervalos
    x = pedidos.sample(n=400, random_state=semilla)["tiempo_entrega_min"]
    m = 1.96 * x.std() / np.sqrt(400)
    filas.append({"inferior": x.mean() - m, "superior": x.mean() + m})
intervalos = pd.DataFrame(filas, index=range(1, 101))
atrapa = (intervalos["inferior"] <= mu) & (mu <= intervalos["superior"])
print(f"Atrapan a μ: {atrapa.sum()} de 100")
print(f"Fallan por debajo: {(intervalos['superior'] < mu).sum()}",
      f"· por encima: {(intervalos['inferior'] > mu).sum()}")

In [ ]:
# [cien_intervalos_grafica]
color = np.where(atrapa, "#8A2E9E", "#F46201")   # naranja: los que fallan
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.hlines(intervalos.index, intervalos["inferior"], intervalos["superior"],
          colors=color, linewidth=2)
ax.axvline(mu, color="#3A0F47", linewidth=2, linestyle="--")
ax.text(mu + 0.1, 103, f"μ = {mu:.2f}", ha="left")
ax.set_ylim(0, 107)
ax.set_title(f"{atrapa.sum()} de los 100 intervalos atrapan μ;"
             f" {(~atrapa).sum()} no")
ax.set_xlabel("Tiempo de entrega (min)")
ax.set_ylabel("Muestra (semilla)")
plt.show()

El 95 % es del **método**: de cada 100 intervalos calculados así, unos 95 atrapan a $\mu$. Un intervalo
ya calculado lo atrapa o no; no tiene «95 % de probabilidad». Que aquí sean 93 y no 95 es azar: con
100 intervalos, lo esperable va de unos 91 a 99.

### Para una proporción

La satisfacción es una proporción: el % de 4 o 5 estrellas. Su error estándar es

$$\mathrm{EE} = \sqrt{\frac{\hat p\,(1-\hat p)}{n}}$$

donde $\hat p$ es la proporción de la muestra y $n$, el número de pedidos **calificados**, no el de
pedidos. La fórmula pide al menos 10 éxitos (4 o 5 estrellas) y 10 fracasos.

In [ ]:
# [proporcion]
calificados = pedidos[pedidos["calificacion"].notna()]
buena = calificados["calificacion"] >= 4   # True si tiene 4 o 5 estrellas
p_hat = buena.mean()                       # la media de True y False
n = len(calificados)
ee_p = np.sqrt(p_hat * (1 - p_hat) / n)
margen_p = 1.96 * ee_p
print(f"n = {n} · éxitos: {buena.sum()} · fracasos: {(~buena).sum()}")
print(f"4 o 5 estrellas: {p_hat * 100:.2f} % ± {margen_p * 100:.2f} puntos")
print(f"Intervalo de 95 %: [{(p_hat - margen_p) * 100:.2f};",
      f"{(p_hat + margen_p) * 100:.2f}] %")

El margen depende de $n$, no de cuántos pedidos hay en total. Estimamos la satisfacción de la
operación, la de los pedidos que vendrán, y no solo la de estos 12 000.

### Un intervalo mide el azar, no el sesgo

Hagamos lo mismo con la insatisfacción, el % de 1 o 2 estrellas.

In [ ]:
# [sesgo_no]
mala = calificados["calificacion"] <= 2    # True si tiene 1 o 2 estrellas
p_mala = mala.mean()
margen_mala = 1.96 * np.sqrt(p_mala * (1 - p_mala) / n)
print(f"1 o 2 estrellas: {p_mala * 100:.2f} %",
      f"± {margen_mala * 100:.2f} puntos")
print(f"Intervalo de 95 %: [{(p_mala - margen_mala) * 100:.2f};",
      f"{(p_mala + margen_mala) * 100:.2f}] %")

En la sesión 5, la comprobación que solo es posible en un caso ficticio dio la verdad: 3,80 % de 1 o 2
estrellas. Queda **fuera** del intervalo. No es mala suerte: es el sesgo de no respuesta (los pedidos
de más de 55 min se califican más). El intervalo mide el azar de quién calificó, no el sesgo. Más
respuestas lo angostan, pero alrededor de la cifra sesgada: el *Literary Digest* tenía más de dos
millones. Ponderar quita el sesgo que depende de lo que vemos, pero no el azar: la cifra ponderada
también lleva su margen. El intervalo no quita nada.

## 4 · La satisfacción, semana a semana

El tablero muestra la satisfacción de cada semana. Con menos calificados, el margen crece. La semana
es la ISO (de lunes a domingo); `isocalendar()` la da como entero sin signo, y la pasamos a `int`
enseguida para poder restar sin sorpresas.

In [ ]:
# [semanal]
fecha = pd.to_datetime(pedidos["fecha"])
semana = fecha.dt.isocalendar().week.astype(int).rename("semana")
grupos = buena.groupby(semana)             # buena: solo los calificados
por_semana = pd.DataFrame({"n": grupos.size(),
                           "% 4 o 5": grupos.mean() * 100})
por_semana["margen"] = 1.96 * np.sqrt(
    por_semana["% 4 o 5"] * (100 - por_semana["% 4 o 5"]) / por_semana["n"])
por_semana["dias"] = fecha.groupby(semana).nunique()   # 7 si está completa
print(f"Semestre: {p_hat * 100:.2f} %")
por_semana.round(1)

In [ ]:
# [semanal_grafica]
fig, ax = plt.subplots(figsize=(9.6, 3.8))
for sem, fila in por_semana.iterrows():
    color = "#F46201" if sem == 13 else "#8A2E9E"
    relleno = color if fila["dias"] == 7 else "white"  # incompleta: hueca
    ax.errorbar(sem, fila["% 4 o 5"], yerr=fila["margen"], fmt="o",
                color=color, mfc=relleno, capsize=3)
    if fila["dias"] < 7 and sem < 14:         # la primera, debajo
        ax.text(sem - 0.4, fila["% 4 o 5"] - fila["margen"] - 1,
                "incompleta", va="top", fontsize=11)
    elif fila["dias"] < 7:                    # la última, a la izquierda
        ax.text(sem - 0.4, fila["% 4 o 5"], "incompleta", ha="right",
                va="center", fontsize=11)
ax.axhline(p_hat * 100, color="#3A0F47", linewidth=2, linestyle="--")
ax.annotate(f"semestre:\n{p_hat * 100:.1f} %", (1.01, p_hat * 100),
            xycoords=("axes fraction", "data"), va="center")
ax.set_title("Cada semana trae unos ±5 puntos de azar")
ax.set_xlabel("Semana ISO (en naranja, la 13)")
ax.set_ylabel("% de 4 o 5 estrellas")
plt.show()

Cada barra es el intervalo de 95 % de su semana. Las semanas completas tienen de 226 a 305
calificados y márgenes de unos ±4 a ±6 puntos; la 27, con dos días, ±11. Solo la semana 13 (en
naranja) tiene un intervalo que no contiene la cifra del semestre.

¿Y por mes? Con unas 1100 a 1300 calificaciones por mes, el margen baja a unos ±2 puntos.

In [ ]:
# [mensual]
mes = fecha.dt.month.rename("mes")
grupos_mes = buena.groupby(mes)
por_mes = pd.DataFrame({"n": grupos_mes.size(),
                        "% 4 o 5": grupos_mes.mean() * 100})
por_mes["margen"] = 1.96 * np.sqrt(
    por_mes["% 4 o 5"] * (100 - por_mes["% 4 o 5"]) / por_mes["n"])
llueve = pedidos["lluvia"] == "Sí"
por_mes["% con lluvia"] = llueve.groupby(mes).mean() * 100
por_mes["tiempo medio"] = tiempo.groupby(mes).mean()
por_mes.round(1)

Abril, el mes con más lluvia, sale más bajo; pero febrero, con poca lluvia, también. El mes solo no
prueba que la lluvia baje la satisfacción: en la sesión 7 mediremos cuántos minutos suma la lluvia.

### La pregunta de entrada: ¿premiamos al equipo de la semana 13?

Su intervalo no contiene la cifra del semestre. Pero con 27 semanas, se esperan 1 o 2 así solo por
azar (el 5 % de 27). Esa semana también fue rápida, y la siguiente bajó al 76,5 %, dentro de su
margen.
**Todavía no**: una semana fuera de su margen pasa; mira si se repite y si el tiempo también cambió.

## 5 · ¿Cuántos calificados hacen falta?

Si se despeja $n$ del margen $E = 1{,}96\,\sqrt{p(1-p)/n}$, sale el tamaño de muestra para un margen
dado:

$$n = \frac{1{,}96^2\, p\,(1-p)}{E^2}$$

El tamaño se redondea siempre **hacia arriba**: con uno menos, el margen pasa de $E$.

In [ ]:
# [tamano]
def tamano(p, e):                          # n = 1,96² · p · (1 − p) / E²
    return np.ceil(1.96 ** 2 * p * (1 - p) / e ** 2)

print(f"Nacional, ±3 puntos: {tamano(p_hat, 0.03):.0f} calificados")
print(f"Nacional, ±5 puntos: {tamano(p_hat, 0.05):.0f} calificados")
tasa = len(calificados) / len(pedidos)     # la parte que califica
for llamadas in [400, 400 * tasa]:
    e = 1.96 * np.sqrt(p_hat * (1 - p_hat) / llamadas) * 100
    print(f"Con {llamadas:.0f} respuestas: ±{e:.1f} puntos")

Si la gerencia llama a 400 pedidos y todos responden, el margen es de unos ±4 puntos. Si responden
como en la app (el 59,2 %), son unas 237 respuestas y unos ±5 puntos, y el sesgo de no respuesta
sigue ahí: el margen no lo mide.

## 6 · Para tu tablero

- El indicador **no cambia**: el % de 4 o 5 estrellas entre los calificados, con su $n$.
- El margen de una proporción pide al menos 10 éxitos y 10 fracasos (OpenIntro 5.1). Con pocos
  calificados, **avisa** o **agrupa** (por ejemplo, por mes), y vuelve a revisar la condición en lo
  agrupado.
- Frente a la cifra del semestre, no anuncies cambios menores que el margen. Y no compares dos
  semanas mirando si sus intervalos se tocan: entre dos semanas, el margen de la diferencia es de
  unos ±7 puntos.
- Las bandas de margen son opcionales: la página del proyecto las pone en «Para ir más lejos
  (opcional)», como «intervalos de confianza para los porcentajes semanales (sesión 6)».

¿Cuántas combinaciones de ciudad y semana cumplen la condición?

In [ ]:
# [ciudad_semana]
por_cs = buena.groupby([calificados["ciudad"], semana])
cs = pd.DataFrame({"n": por_cs.size(), "exitos": por_cs.sum()})
cs["fracasos"] = cs["n"] - cs["exitos"]
cs["aplica"] = (cs["exitos"] >= 10) & (cs["fracasos"] >= 10)
print(f"Combinaciones de ciudad y semana: {len(cs)}",
      f"· con 10 éxitos y 10 fracasos: {cs['aplica'].sum()}")
por_ciudad = cs.groupby(level="ciudad")
pd.DataFrame({"calificados por semana (mediana)": por_ciudad["n"].median(),
              "semanas con 10 y 10": por_ciudad["aplica"].sum()})

En Bogotá todas las semanas cumplen; en Barranquilla y Bucaramanga, ninguna. Una semana de
Bucaramanga con pocos calificados, todos de 4 o 5, da 100 % con margen 0: la fórmula no aplica ahí.
Agrupar por mes casi siempre basta, pero no siempre: revisa la condición también en los meses.

---
# Práctica guiada

Tres ejercicios; el tercero es **opcional** (o para la casa). Cada celda termina con un verificador
que dice qué está bien y, en lo demás, te da una pista sin darte la respuesta. Completa lo que se
indica y ejecuta; puedes repetir la celda las veces que quieras. Las respuestas escritas van al
final, en «Tus respuestas»: esas no las revisa ningún verificador.

## Ejercicio 1 · La distribución muestral con $n = 100$

Repite la celda [mil_muestras] con muestras de **100** pedidos y tu propia semilla.

- Cambia `mi_semilla` por un número de tres cifras que elijas tú.
- La plantilla trae el tamaño de la clase: cámbialo por el que pide el ejercicio.
- A la línea de `medias.append` le falta algo: el ejercicio pide la **media** de cada muestra.
- Esta celda reemplaza la lista `medias` de la clase: si después vuelves a una celda de la clase, usa
  **Entorno de ejecución → Ejecutar anteriores**.

Después responde abajo: el verificador te dice tu EE. ¿Cuánto quedaría con $n = 400$, y por qué?

In [ ]:
# [ej1]
mi_semilla = 123   # cámbiala por un número de tres cifras que elijas tú
medias = []
for i in range(1000):
    # ← cambia el 400 por el tamaño que pide el ejercicio
    muestra = pedidos.sample(n=400, random_state=mi_semilla + i)
    medias.append(muestra["tiempo_entrega_min"])   # ← ¿qué le falta?
revisar_medias(medias)

## Ejercicio 2 · El intervalo de la semana 20

La semana 20 es la semana completa con el % de 4 o 5 estrellas más bajo. Calcula su intervalo de
95 %, **en %**: la proporción $\hat p$ entre sus calificados, su $n$, su EE y el intervalo. La variable
`semana` ya existe: viene de [semanal]. Pista: las celdas [proporcion] y [semanal].

Deja como está la llamada final: entrega cada extremo con su nombre.

Después responde abajo: ¿fue la semana 20 peor que el semestre?

In [ ]:
# [ej2]
semana_20 = pedidos[semana == 20]   # los pedidos de la semana 20
p_20 = ...       # la proporción de 4 o 5 estrellas entre sus calificados
n_20 = ...       # cuántos calificados tiene
ee_20 = ...      # su error estándar
inferior = ...   # el intervalo de 95 %, en %
superior = ...
revisar_intervalo(inferior=inferior, superior=superior)

## Ejercicio 3 (opcional) · ¿Cuántos calificados necesita Bucaramanga para ±5 puntos?

Con la $\hat p$ de 4 o 5 estrellas de Bucaramanga (entre sus calificados), calcula cuántos calificados
hacen falta para un margen de ±5 puntos. Pista: la celda [tamano]. Si escribes $\hat p$ en proporción,
$E$ también va en proporción (`0.05`).

Después responde abajo: ¿cuántas semanas de calificaciones de Bucaramanga son? (Tiene unos 27
calificados por semana.) ¿Conviene mostrarla por semana o por mes?

In [ ]:
# [ej3]
bucaramanga = ...   # los calificados de Bucaramanga
p_buc = ...         # su proporción de 4 o 5 estrellas
n_necesario = ...   # el tamaño para un margen de ±5 puntos
revisar_tamano(n=n_necesario)

### Tus respuestas

*Doble clic para escribir aquí.*

1. Ejercicio 1: mi EE con $n = 100$ salió …; con $n = 400$ quedaría cerca de … porque …
2. Ejercicio 2: el intervalo de la semana 20 es [… ; …]. ¿Fue peor que el semestre (79,07 %)? …
   porque …
3. Ejercicio 3: Bucaramanga necesita … calificados, que son unas … semanas; conviene mostrarla …
   porque …
4. Autoevaluación. ¿Verdadero o falso, o qué harías? Explica cada una.
   - Una semana de Bucaramanga trae 4 calificaciones, todas de 4 o 5: 100 % ± 0. ¿Lo publicas así? …
   - «El 95 % de los pedidos tarda entre 33,2 y 35,5 min.» …
   - «Una encuesta voluntaria con 100 000 respuestas tiene ±0,25 puntos de margen y aun así puede
     estar lejos de la verdad.» …
   - Del tiempo de entrega tienes dos cifras: 12,7 min y 0,62 min (de muestras de 400). ¿Cuál usas
     para decir cuánto varía un pedido y cuál para decir cuánto confiar en la media? …
   - «Con el doble de calificados, el margen se parte por la mitad.» …

---
## La respuesta a la gerencia

**¿Cuánto confiar en una muestra? Lo que diga su margen.** El 79,1 % de 4 o 5 estrellas del semestre,
con 7099 calificados, tiene un margen de ±1 punto; una semana, con unos 270 calificados, de unos ±5;
y en una semana de Bucaramanga la fórmula ni siquiera aplica: hay que agrupar por mes o avisar. Y el
margen no cubre el sesgo: que los calificados representen a todos sigue siendo el supuesto de la
sesión 5.